In [ ]:
!pip install catboost xgboost lightgbm torch pandas numpy scikit-learn tqdm matplotlib joblib

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter, defaultdict
from scipy import stats
from scipy.stats import skew, kurtosis, norm, shapiro

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (16, 8)
plt.rcParams["font.size"] = 11

# === Kaggle paths (EDIT nếu dataset khác tên) ===
DATA_DIR = "//kaggle/input/datasets/trmnguyntrngbch/dataeda"   # <- đổi nếu dataset name khác
FIGURES_DIR = "/kaggle/working/figures"
OUTPUT_DIR = "/kaggle/working/output"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

HORIZON = 28
TRAIN_LAST_D = None

SAVE = lambda name: plt.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight")

print("\n" + "=" * 90)
print(" " * 20 + "M5 FORECASTING - COMPREHENSIVE EDA")
print("=" * 90)
print("Sections: Statistics | Distribution | Trends | Seasonality | Events | Prices | SNAP")
print("          Data Quality | Hierarchy | Sparsity | Outliers | Autocorrelation | Features")
print("=" * 90)

# ============================================================================
# [1] LOAD DATA - Comprehensive metrics collection
# ============================================================================
print("\n[1] LOADING & ANALYZING DATA...")

CHUNK = 1_000_000
DTYPES = {
    "item_id":"category", "dept_id":"category", "cat_id":"category",
    "store_id":"category", "state_id":"category", "d":"category",
    "day_id":"uint16", "wm_yr_wk":"uint16", "sales":"float32",
    "sell_price":"float32", "wday":"uint8", "month":"uint8",
    "year":"uint16", "snap_CA":"uint8", "snap_TX":"uint8", "snap_WI":"uint8"
}

# === Initialize comprehensive counters ===
nrows_total = 0
sales_sum = sales_sq_sum = sales_cubed_sum = 0.0
sales_min, sales_max = float("inf"), float("-inf")
zero_count = positive_count = 0

# Store-level
store_sales = Counter()
store_zero = Counter()
store_items = defaultdict(set)
store_dates = defaultdict(set)

# Item-level
item_sales = Counter()
item_zero = Counter()
item_max = Counter()
item_min = Counter()
item_dates = defaultdict(set)
item_stores = defaultdict(set)
item_nonzero_days = Counter()

# Department & Category
dept_sales = Counter()
cat_sales = Counter()
state_sales = Counter()

# Temporal
dow_sales = Counter()
dow_cnt = Counter()
month_sales = Counter()
month_cnt = Counter()
year_sales = Counter()

# Events
event_sales_sum = event_sales_cnt = 0
noevent_sales_sum = noevent_sales_cnt = 0
event_type_1_cnt = Counter()
event_type_2_cnt = Counter()
event_name_1_cnt = Counter()
event_name_2_cnt = Counter()

# Price
price_sum = price_sq_sum = price_cnt = 0
price_min = float("inf")
price_max = float("-inf")
price_by_item = defaultdict(list)

# Missing per-column
missing_counts = Counter()

# Sampling
sampled_rows = []
SAMPLE_N = 250_000

LOG_EVERY = 1  # in log mỗi chunk

print(f"  -> DATA_DIR = {DATA_DIR}")
print("  -> Start loading train.csv ...")

for chunk_idx, chunk in enumerate(pd.read_csv(
    os.path.join(DATA_DIR, "train.csv"),
    chunksize=CHUNK, dtype=DTYPES, parse_dates=["date"]
)):
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"\n  [Chunk {chunk_idx+1}] read {len(chunk):,} rows")

    # ======= MISSING =======
    t0 = pd.Timestamp.now()
    miss = chunk.isna().sum()
    for k, v in miss.items():
        missing_counts[k] += int(v)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - missing count done ({(t1-t0).total_seconds():.2f}s)")

    # ======= SALES BASIC =======
    t0 = pd.Timestamp.now()
    n = len(chunk)
    nrows_total += n

    sales = chunk["sales"].values
    sales_sum += sales.sum()
    sales_sq_sum += (sales ** 2).sum()
    sales_cubed_sum += (sales ** 3).sum()
    sales_min = min(sales_min, sales.min())
    sales_max = max(sales_max, sales.max())

    zero_mask = sales == 0
    zero_count += zero_mask.sum()
    positive_count += (~zero_mask).sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - sales/basic done ({(t1-t0).total_seconds():.2f}s)")

    # ======= ITEM LEVEL =======
    t0 = pd.Timestamp.now()
    for item_id in chunk["id"].unique():
        item_chunk = chunk[chunk["id"] == item_id]
        item_s = item_chunk["sales"].sum()
        item_sales[item_id] += item_s
        item_zero[item_id] += int((item_chunk["sales"] == 0).sum())
        item_max[item_id] = max(item_max.get(item_id, 0), item_chunk["sales"].max())
        item_min[item_id] = min(item_min.get(item_id, float("inf")), item_chunk["sales"].min())
        item_nonzero_days[item_id] += int((item_chunk["sales"] > 0).sum())
        item_dates[item_id].update(item_chunk["date"].values)
        item_stores[item_id].update(item_chunk["store_id"].values)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - item level done ({(t1-t0).total_seconds():.2f}s)")

    # ======= STORE LEVEL =======
    t0 = pd.Timestamp.now()
    for store_id in chunk["store_id"].unique():
        store_chunk = chunk[chunk["store_id"] == store_id]
        store_sales[store_id] += store_chunk["sales"].sum()
        store_zero[store_id] += int((store_chunk["sales"] == 0).sum())
        store_items[store_id].update(store_chunk["id"].unique())
        store_dates[store_id].update(store_chunk["date"].values)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - store level done ({(t1-t0).total_seconds():.2f}s)")

    # ======= DEPT/CAT/STATE =======
    t0 = pd.Timestamp.now()
    for dept_id in chunk["dept_id"].unique():
        dept_chunk = chunk[chunk["dept_id"] == dept_id]
        dept_sales[dept_id] += dept_chunk["sales"].sum()

    for cat_id in chunk["cat_id"].unique():
        cat_chunk = chunk[chunk["cat_id"] == cat_id]
        cat_sales[cat_id] += cat_chunk["sales"].sum()

    for state_id in chunk["state_id"].unique():
        state_chunk = chunk[chunk["state_id"] == state_id]
        state_sales[state_id] += state_chunk["sales"].sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - dept/cat/state done ({(t1-t0).total_seconds():.2f}s)")

    # ======= TEMPORAL =======
    t0 = pd.Timestamp.now()
    for wday in chunk["wday"].unique():
        dow_chunk = chunk[chunk["wday"] == wday]
        dow_sales[wday] += dow_chunk["sales"].sum()
        dow_cnt[wday] += len(dow_chunk)

    for month in chunk["month"].unique():
        month_chunk = chunk[chunk["month"] == month]
        month_sales[month] += month_chunk["sales"].sum()
        month_cnt[month] += len(month_chunk)

    for year in chunk["year"].unique():
        year_chunk = chunk[chunk["year"] == year]
        year_sales[year] += year_chunk["sales"].sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - temporal done ({(t1-t0).total_seconds():.2f}s)")

    # ======= EVENTS =======
    t0 = pd.Timestamp.now()
    for et1 in chunk["event_type_1"].unique():
        if pd.notna(et1) and et1 != "None":
            et1_chunk = chunk[chunk["event_type_1"] == et1]
            event_type_1_cnt[et1] += len(et1_chunk)

    for et2 in chunk["event_type_2"].unique():
        if pd.notna(et2) and et2 != "None":
            et2_chunk = chunk[chunk["event_type_2"] == et2]
            event_type_2_cnt[et2] += len(et2_chunk)

    for en1 in chunk["event_name_1"].unique():
        if pd.notna(en1) and en1 != "None":
            en1_chunk = chunk[chunk["event_name_1"] == en1]
            event_name_1_cnt[en1] += len(en1_chunk)

    for en2 in chunk["event_name_2"].unique():
        if pd.notna(en2) and en2 != "None":
            en2_chunk = chunk[chunk["event_name_2"] == en2]
            event_name_2_cnt[en2] += len(en2_chunk)

    event_mask = (chunk["event_type_1"] != "None") | (chunk["event_type_2"] != "None")
    event_sales_sum += chunk.loc[event_mask, "sales"].sum()
    event_sales_cnt += event_mask.sum()
    noevent_sales_sum += chunk.loc[~event_mask, "sales"].sum()
    noevent_sales_cnt += (~event_mask).sum()
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - events done ({(t1-t0).total_seconds():.2f}s)")

    # ======= PRICE =======
    t0 = pd.Timestamp.now()
    p = chunk["sell_price"].dropna().values
    price_sum += p.sum()
    price_sq_sum += (p ** 2).sum()
    price_cnt += len(p)
    if len(p) > 0:
        price_min = min(price_min, p.min())
        price_max = max(price_max, p.max())

    for item_id in chunk["id"].unique():
        item_prices = chunk[chunk["id"] == item_id]["sell_price"].dropna().values
        if len(item_prices) > 0:
            price_by_item[item_id].extend(item_prices)
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - price done ({(t1-t0).total_seconds():.2f}s)")

    # ======= SAMPLING =======
    t0 = pd.Timestamp.now()
    if len(sampled_rows) < SAMPLE_N:
        idx = np.random.choice(n, size=min(n, SAMPLE_N - len(sampled_rows)), replace=False)
        sampled_rows.append(chunk.iloc[idx])
    t1 = pd.Timestamp.now()
    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    - sampling done ({(t1-t0).total_seconds():.2f}s)")

    if (chunk_idx + 1) % LOG_EVERY == 0:
        print(f"    > total so far: rows={nrows_total:,} | items={len(item_sales):,} | zeros={zero_count:,}")

print(f"\n  ✓ Done train.csv: {nrows_total:,} rows loaded in {chunk_idx+1} chunks")

print("  -> Start loading val.csv ...")
val_shape = sum(1 for _ in open(os.path.join(DATA_DIR, "val.csv"))) - 1
val = pd.read_csv(os.path.join(DATA_DIR, "val.csv"), dtype=DTYPES, parse_dates=["date"])
print(f"  ✓ Done val.csv: {val_shape:,} rows")

print("  -> Start loading test.csv ...")
test_shape = sum(1 for _ in open(os.path.join(DATA_DIR, "test.csv"))) - 1
print(f"  ✓ Done test.csv: {test_shape:,} rows")

train_sample = pd.concat(sampled_rows, ignore_index=True)
n_items = len(item_sales)
n_stores = len(store_sales)
n_depts = len(dept_sales)
n_cats = len(cat_sales)
n_states = len(state_sales)

stores_sorted = sorted(store_sales.keys())
items_sorted = sorted(item_sales.keys())
depts_sorted = sorted(dept_sales.keys())
cats_sorted = sorted(cat_sales.keys())

print(f"  ✓ Items: {n_items:,} | Stores: {n_stores} | Depts: {n_depts} | Categories: {n_cats} | States: {n_states}")
print(f"  ✓ Train: {nrows_total:,} | Val: {val_shape:,} | Test: {test_shape:,}")
print(f"  ✓ Sample: {len(train_sample):,} rows")

# ============================================================================
# [2] BASIC STATISTICS & DATA QUALITY
# ============================================================================
print("\n[2] STATISTICS & DATA QUALITY...")

sales_mean = sales_sum / nrows_total if nrows_total > 0 else 0
sales_var = (sales_sq_sum / nrows_total - sales_mean ** 2) if nrows_total > 0 else 0
sales_std = np.sqrt(max(sales_var, 0))
sales_skew_est = sales_cubed_sum / nrows_total / (sales_std ** 3) if sales_std > 0 else 0
zero_pct = zero_count / nrows_total * 100 if nrows_total > 0 else 0
sparsity = zero_pct / 100

price_mean = price_sum / price_cnt if price_cnt > 0 else 0
price_var = price_sq_sum / price_cnt - price_mean ** 2 if price_cnt > 0 else 0
price_std = np.sqrt(max(price_var, 0))

# Event impact
base_avg = noevent_sales_sum / max(noevent_sales_cnt, 1) if noevent_sales_cnt > 0 else 0
event_avg = event_sales_sum / max(event_sales_cnt, 1) if event_sales_cnt > 0 else 0
event_lift = (event_avg / base_avg - 1) * 100 if base_avg > 0 else 0

# Item metrics
items_zero_pct = np.array([item_zero[i] / max(item_nonzero_days[i] + item_zero[i], 1) * 100 
                           for i in items_sorted])
items_velocity = np.array([item_nonzero_days[i] / max(item_zero[i] + item_nonzero_days[i], 1) 
                           for i in items_sorted])

print(f"\n  Sales Statistics:")
print(f"    Mean:           {sales_mean:12.4f}")
print(f"    Std Dev:        {sales_std:12.4f}")
print(f"    Min:            {sales_min:12.4f}")
print(f"    Max:            {sales_max:12.4f}")
print(f"    Skewness (est): {sales_skew_est:12.4f}")
print(f"    CV (Coeff Var): {sales_std/sales_mean:12.4f}" if sales_mean > 0 else "")

print(f"\n  Sparsity & Zero Sales:")
print(f"    Zero counts:    {zero_count:13,} ({zero_pct:6.2f}%)")
print(f"    Positive:       {positive_count:13,} ({100-zero_pct:6.2f}%)")

print(f"\n  Pricing:")
print(f"    Mean:           {price_mean:12.4f}")
print(f"    Std Dev:        {price_std:12.4f}")
print(f"    Min:            {price_min:12.4f}")
print(f"    Max:            {price_max:12.4f}")
print(f"    Records:        {price_cnt:13,}")

print(f"\n  Events Impact:")
print(f"    Avg w/o event:  {base_avg:12.4f}")
print(f"    Avg w/ event:   {event_avg:12.4f}")
print(f"    Lift:           {event_lift:12.2f}%")

print(f"\n  Items Characteristics:")
print(f"    Items (total):  {len(item_sales):13,}")
print(f"    Items w/ sales: {sum(1 for v in item_nonzero_days.values() if v > 0):13,}")
print(f"    Avg stores/item:{np.mean([len(item_stores[i]) for i in items_sorted]):13.1f}")
print(f"    Avg zero % (item): {items_zero_pct.mean():12.2f}%")
print(f"    Velocity (avg): {items_velocity.mean():12.4f}")

# ============================================================================
# [3] DISTRIBUTION ANALYSIS
# ============================================================================
print("\n[3] SALES DISTRIBUTION...")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 3.1: Log distribution
ax = axes[0, 0]
s_nz = train_sample[train_sample["sales"] > 0]["sales"]
ax.hist(np.log1p(s_nz), bins=100, edgecolor="white", linewidth=0.3, alpha=0.7, color="steelblue")
ax.axvline(np.log1p(s_nz.median()), color="red", ls="--", lw=2, label=f"Median={np.log1p(s_nz.median()):.2f}")
ax.axvline(np.log1p(s_nz.mean()), color="orange", ls="--", lw=2, label=f"Mean={np.log1p(s_nz.mean()):.2f}")
ax.set_xlabel("log(1+sales)"); ax.set_ylabel("Frequency")
ax.set_title("Sales Distribution (log, non-zero)", fontweight="bold", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 3.2: Raw distribution (first 50 units)
ax = axes[0, 1]
ax.hist(s_nz[s_nz <= 50], bins=50, edgecolor="white", linewidth=0.5, alpha=0.7, color="coral")
ax.set_xlabel("Sales (units)"); ax.set_ylabel("Frequency")
ax.set_title("Sales Distribution (0-50 units)", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3)

# 3.3: Q-Q plot for normality check
ax = axes[0, 2]
log_sales = np.log1p(s_nz.values)
stats.probplot(log_sales, dist="norm", plot=ax)
ax.set_title(f"Q-Q Plot (log sales)\nSkew={skew(log_sales):.2f}", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3)

# 3.4: Store-wise zero percentage
ax = axes[1, 0]
n_per_store = nrows_total // n_stores
zero_pcts = [store_zero[s] / n_per_store * 100 for s in stores_sorted]
bars = ax.bar(range(len(stores_sorted)), zero_pcts, color="lightcoral", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(stores_sorted)))
ax.set_xticklabels(stores_sorted, rotation=0)
ax.set_ylabel("Zero Sales %"); ax.set_title("Zero Sales % by Store", fontweight="bold", fontsize=12)
ax.axhline(np.mean(zero_pcts), color="red", ls="--", lw=1, label=f"Avg: {np.mean(zero_pcts):.1f}%")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

# 3.5: Mean sales by store
ax = axes[1, 1]
means_store = [store_sales[s] / (nrows_total // n_stores) for s in stores_sorted]
bars = ax.bar(range(len(stores_sorted)), means_store, color="mediumseagreen", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(stores_sorted)))
ax.set_xticklabels(stores_sorted, rotation=0)
ax.set_ylabel("Mean Daily Sales"); ax.set_title("Avg Daily Sales by Store", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3, axis="y")

# 3.6: Mean sales by category
ax = axes[1, 2]
cat_avg = {c: cat_sales[c] / max(1, nrows_total // n_cats) for c in cats_sorted}
cats_sorted_by_sales = sorted(cat_avg.keys(), key=lambda c: cat_avg[c], reverse=True)
cat_vals = [cat_avg[c] for c in cats_sorted_by_sales]
bars = ax.bar(range(len(cat_vals)), cat_vals, color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(cats_sorted_by_sales)))
ax.set_xticklabels(cats_sorted_by_sales, rotation=45, ha="right")
ax.set_ylabel("Mean Daily Sales"); ax.set_title("Avg Daily Sales by Category", fontweight="bold", fontsize=12)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("01_distribution.png")
plt.close()
print("  ✓ Saved: 01_distribution.png")

# ============================================================================
# [4] TREND ANALYSIS
# ============================================================================
print("\n[4] TREND ANALYSIS...")

train_sample["date"] = pd.to_datetime(train_sample["date"])
daily = train_sample.groupby("date")["sales"].agg(["sum", "mean", "std", "count"]).reset_index()
daily.columns = ["date", "total_sales", "mean_sales", "std_sales", "n_records"]

val_daily = val.groupby("date")["sales"].agg(["sum", "mean"]).reset_index()
val_daily.columns = ["date", "total_sales", "mean_sales"]

fig, axes = plt.subplots(3, 2, figsize=(20, 15))

# 4.1: Total sales over time with MA
ax = axes[0, 0]
ax.plot(daily["date"], daily["total_sales"], lw=0.5, color="steelblue", alpha=0.5, label="Daily Sales")
ax.plot(daily["date"], daily["total_sales"].rolling(7, center=True).mean(), 
        color="darkblue", lw=1.5, label="7-day MA")
ax.plot(daily["date"], daily["total_sales"].rolling(30, center=True).mean(), 
        color="red", lw=1.5, label="30-day MA")
ax.plot(val_daily["date"], val_daily["total_sales"], lw=0.6, color="green", alpha=0.6, label="Val (2016)")
ax.axvline(pd.Timestamp("2016-04-25"), color="black", ls="--", lw=1, alpha=0.5)
ax.set_ylabel("Total Sales"); ax.set_title("Daily Sales Trend with Moving Averages", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4.2: Sales volatility (rolling std)
ax = axes[0, 1]
ax.plot(daily["date"], daily["std_sales"], lw=0.7, color="orange", alpha=0.7)
ax.plot(daily["date"], daily["std_sales"].rolling(30).mean(), color="red", lw=2, label="30-day MA")
ax.set_ylabel("Std Dev of Sales"); ax.set_title("Sales Volatility Over Time", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4.3: Year-over-year
ax = axes[1, 0]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
for year_idx, year in enumerate([2011, 2012, 2013, 2014, 2015, 2016]):
    y = daily[daily["date"].dt.year == year].copy()
    if not y.empty:
        y["doy"] = y["date"].dt.dayofyear
        ax.plot(y["doy"], y["total_sales"], lw=1.2, alpha=0.8, label=str(year), color=colors[year_idx])
ax.set_xlabel("Day of Year"); ax.set_ylabel("Total Sales")
ax.set_title("Year-over-Year Comparison", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4.4: Growth rate
ax = axes[1, 1]
daily_sorted = daily.sort_values("date").reset_index(drop=True)
growth = daily_sorted["total_sales"].pct_change().rolling(7).mean() * 100
ax.plot(daily_sorted["date"], growth, lw=0.8, color="purple")
ax.axhline(0, color="black", ls="-", lw=0.5)
ax.fill_between(daily_sorted["date"], growth, 0, where=(growth >= 0), alpha=0.3, color="green")
ax.fill_between(daily_sorted["date"], growth, 0, where=(growth < 0), alpha=0.3, color="red")
ax.set_ylabel("Growth Rate (%)"); ax.set_title("7-day Rolling Growth Rate", fontweight="bold")
ax.grid(alpha=0.3)

# 4.5: Monthly totals
ax = axes[2, 0]
monthly = train_sample.copy()
monthly["ym"] = monthly["date"].dt.to_period("M")
monthly_sales = monthly.groupby("ym")["sales"].sum()
ax.bar(range(len(monthly_sales)), monthly_sales.values, color="steelblue", edgecolor="black", alpha=0.7)
ax.set_ylabel("Total Sales"); ax.set_title("Monthly Sales Aggregates", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 4.6: State-wise trends
ax = axes[2, 1]
for state in sorted(state_sales.keys()):
    state_data = train_sample[train_sample["state_id"] == state]
    state_daily = state_data.groupby("date")["sales"].sum().rolling(7).mean()
    ax.plot(state_daily.index, state_daily.values, lw=1, label=state, alpha=0.8)
ax.set_ylabel("7-day MA Sales"); ax.set_title("State-wise Sales Trends", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
SAVE("02_trends.png")
plt.close()
print("  ✓ Saved: 02_trends.png")

# ============================================================================
# [5] SEASONALITY ANALYSIS
# ============================================================================
print("\n[5] SEASONALITY...")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 5.1: Day of week
ax = axes[0, 0]
dow_labels = ["Sat", "Sun", "Mon", "Tue", "Wed", "Thu", "Fri"]
dow_n = [dow_cnt.get(d, 1) for d in range(1, 8)]
dow_vals = [dow_sales.get(d, 0) / max(n, 1) for d, n in zip(range(1, 8), dow_n)]
colors_dow = ["salmon" if d in [1, 2] else "skyblue" for d in range(1, 8)]
bars = ax.bar(dow_labels, dow_vals, color=colors_dow, edgecolor="black", linewidth=1)
ax.set_ylabel("Avg Sales"); ax.set_title("Day of Week Effect", fontweight="bold")
for bar, v in zip(bars, dow_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(dow_vals)*0.02, 
            f"{v:.0f}", ha="center", fontsize=10, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.2: Month
ax = axes[0, 1]
month_vals = [month_sales.get(m, 0) for m in range(1, 13)]
colors_m = ["#FFB3BA" if m in [10, 11, 12] else "#BAE1FF" for m in range(1, 13)]
labels_m = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
bars = ax.bar(labels_m, month_vals, color=colors_m, edgecolor="black", linewidth=1)
ax.set_ylabel("Total Sales"); ax.set_title("Seasonal Pattern by Month", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.3: Weekend vs Weekday
ax = axes[0, 2]
w_avg = sum(dow_sales.get(d, 0) for d in [1, 2]) / max(sum(dow_cnt.get(d, 1) for d in [1, 2]), 1)
d_avg = sum(dow_sales.get(d, 0) for d in [3, 4, 5, 6, 7]) / max(sum(dow_cnt.get(d, 1) for d in [3, 4, 5, 6, 7]), 1)
bars = ax.bar(["Weekend", "Weekday"], [w_avg, d_avg], color=["salmon", "skyblue"], 
              edgecolor="black", width=0.5, linewidth=1)
ax.set_ylabel("Avg Daily Sales"); ax.set_title("Weekend vs Weekday", fontweight="bold")
for bar, v in zip(bars, [w_avg, d_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max([w_avg, d_avg])*0.02, 
            f"{v:.0f}", ha="center", fontsize=11, fontweight="bold")
wk_ratio = (w_avg / d_avg - 1) * 100 if d_avg > 0 else 0
ax.text(0.5, max([w_avg, d_avg]) * 0.5, f"Diff: {wk_ratio:+.1f}%", ha="center", fontsize=12, 
        bbox=dict(boxstyle="round", facecolor="yellow", alpha=0.3), fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.4: Event impact
ax = axes[1, 0]
bars = ax.bar(["No Event", "Has Event"], [base_avg, event_avg], 
              color=["lightgray", "goldenrod"], edgecolor="black", width=0.5, linewidth=1)
ax.set_ylabel("Avg Daily Sales"); ax.set_title(f"Event Impact (Lift: {event_lift:+.1f}%)", fontweight="bold")
for bar, v in zip(bars, [base_avg, event_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max([base_avg, event_avg])*0.02, 
            f"{v:.0f}", ha="center", fontsize=11, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 5.5: Autocorrelation
ax = axes[1, 1]
daily_sales_for_acf = daily.sort_values("date")["total_sales"].values
lags = np.arange(1, 61)
acf_values = [np.corrcoef(daily_sales_for_acf[:-lag], daily_sales_for_acf[lag:])[0, 1] for lag in lags]
ax.plot(lags, acf_values, lw=1.5, color="steelblue", marker="o", markersize=3)
ax.axhline(0, color="black", ls="-", lw=0.5)
ax.axhline(1.96/np.sqrt(len(daily_sales_for_acf)), color="red", ls="--", lw=1, label="95% CI")
ax.axhline(-1.96/np.sqrt(len(daily_sales_for_acf)), color="red", ls="--", lw=1)
ax.set_xlabel("Lag (days)"); ax.set_ylabel("Autocorrelation"); ax.set_title("Autocorrelation (ACF)", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 5.6: Quarterly pattern
ax = axes[1, 2]
daily["quarter"] = daily["date"].dt.quarter
quarterly = daily.groupby("quarter")["total_sales"].sum()
quarters = ["Q1", "Q2", "Q3", "Q4"]
bars = ax.bar(quarters[:len(quarterly)], quarterly.values, color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_ylabel("Total Sales"); ax.set_title("Quarterly Sales Pattern", fontweight="bold")
for bar, v in zip(bars, quarterly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + quarterly.max()*0.02, 
            f"{v/1e6:.1f}M", ha="center", fontsize=10, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("03_seasonality.png")
plt.close()
print("  ✓ Saved: 03_seasonality.png")

# ============================================================================
# [6] EVENT & SNAP ANALYSIS
# ============================================================================
print("\n[6] EVENTS & SNAP...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 6.1: Event types
ax = axes[0, 0]
et_counts = sorted(event_type_1_cnt.items(), key=lambda x: x[1], reverse=True)[:10]
if et_counts:
    names, counts = zip(*et_counts)
    ax.barh(names, counts, color="steelblue", edgecolor="black", linewidth=1)
    ax.set_xlabel("Frequency"); ax.set_title("Top 10 Event Types 1", fontweight="bold")
    ax.grid(alpha=0.3, axis="x")

# 6.2: Event names
ax = axes[0, 1]
en_counts = sorted(event_name_1_cnt.items(), key=lambda x: x[1], reverse=True)[:10]
if en_counts:
    names, counts = zip(*en_counts)
    ax.barh(names, counts, color="coral", edgecolor="black", linewidth=1)
    ax.set_xlabel("Frequency"); ax.set_title("Top 10 Event Names 1", fontweight="bold")
    ax.grid(alpha=0.3, axis="x")

# 6.3: SNAP impact by state
ax = axes[1, 0]
snap_cols = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
snap_data = {}
for state, snap_col in snap_cols.items():
    state_df = train_sample[train_sample["state_id"] == state]
    snap_0 = state_df[state_df[snap_col] == 0]["sales"].mean()
    snap_1 = state_df[state_df[snap_col] == 1]["sales"].mean()
    snap_data[state] = (snap_0, snap_1)

x = np.arange(len(snap_data))
width = 0.35
bars1 = ax.bar(x - width/2, [v[0] for v in snap_data.values()], width, label="No SNAP", 
               color="lightcoral", edgecolor="black", linewidth=1)
bars2 = ax.bar(x + width/2, [v[1] for v in snap_data.values()], width, label="SNAP", 
               color="mediumseagreen", edgecolor="black", linewidth=1)
ax.set_ylabel("Avg Sales"); ax.set_title("SNAP Program Impact by State", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(snap_data.keys())
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

for i, (state, (s0, s1)) in enumerate(snap_data.items()):
    lift = (s1 / s0 - 1) * 100 if s0 > 0 else 0
    ax.text(i, max(s0, s1) * 1.05, f"{lift:+.1f}%", ha="center", fontsize=10, fontweight="bold")

# 6.4: Event frequency by month
ax = axes[1, 1]
event_monthly = train_sample[train_sample["event_type_1"] != "None"].copy()
if len(event_monthly) > 0:
    event_monthly["month_name"] = pd.to_datetime(event_monthly["date"]).dt.strftime("%b")
    event_cnt = event_monthly.groupby("month_name").size()
    months_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    event_cnt = event_cnt.reindex([m for m in months_order if m in event_cnt.index])
    ax.bar(range(len(event_cnt)), event_cnt.values, color="goldenrod", edgecolor="black", linewidth=1)
    ax.set_xticks(range(len(event_cnt))); ax.set_xticklabels(event_cnt.index, rotation=45)
    ax.set_ylabel("Event Count"); ax.set_title("Event Frequency by Month", fontweight="bold")
    ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("04_events_snap.png")
plt.close()
print("  ✓ Saved: 04_events_snap.png")

# ============================================================================
# [7] PRICE ANALYSIS
# ============================================================================
print("\n[7] PRICE ANALYSIS...")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 7.1: Price distribution
ax = axes[0, 0]
prices = train_sample["sell_price"].dropna()
ax.hist(prices, bins=100, edgecolor="white", alpha=0.7, color="mediumblue")
ax.axvline(prices.mean(), color="red", ls="--", lw=2, label=f"Mean: {prices.mean():.2f}")
ax.axvline(prices.median(), color="orange", ls="--", lw=2, label=f"Median: {prices.median():.2f}")
ax.set_xlabel("Price ($)"); ax.set_ylabel("Frequency")
ax.set_title("Sell Price Distribution", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 7.2: Price by category
ax = axes[0, 1]
cat_price = train_sample.groupby("cat_id")["sell_price"].agg(["mean", "std"]).sort_values("mean", ascending=False)
x = np.arange(len(cat_price))
ax.bar(x, cat_price["mean"], yerr=cat_price["std"], color="steelblue", edgecolor="black", 
       linewidth=1, capsize=5, alpha=0.7, error_kw={"linewidth": 2})
ax.set_xticks(x); ax.set_xticklabels(cat_price.index, rotation=0)
ax.set_ylabel("Price ($)"); ax.set_title("Mean Price by Category (with std)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 7.3: Price volatility (top items)
ax = axes[0, 2]
item_price_vol = {}
for item_id in train_sample["id"].unique():
    prices_item = train_sample[train_sample["id"] == item_id]["sell_price"].dropna()
    if len(prices_item) > 0:
        item_price_vol[item_id] = prices_item.std()
top_vol = sorted(item_price_vol.items(), key=lambda x: x[1], reverse=True)[:15]
names, vols = zip(*top_vol)
ax.barh(range(len(names)), vols, color="darkorange", edgecolor="black", linewidth=1)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel("Price Std Dev ($)"); ax.set_title("Top 15 Items by Price Volatility", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# 7.4: Price-Sales correlation
ax = axes[1, 0]
sample_for_corr = train_sample[["sales", "sell_price"]].dropna()
corr_coef = sample_for_corr["sales"].corr(sample_for_corr["sell_price"])
ax.scatter(sample_for_corr["sell_price"], sample_for_corr["sales"], alpha=0.3, s=10, color="steelblue")
ax.set_xlabel("Price ($)"); ax.set_ylabel("Sales (units)")
ax.set_title(f"Price-Sales Relationship (corr: {corr_coef:.4f})", fontweight="bold")
ax.grid(alpha=0.3)

# 7.5: Price changes over time
ax = axes[1, 1]
for item in list(train_sample["id"].unique())[:5]:  # Top 5 items
    item_data = train_sample[train_sample["id"] == item].sort_values("date")
    if len(item_data) > 100:
        prices_rolling = item_data.groupby(item_data["date"].dt.to_period("M"))["sell_price"].mean()
        ax.plot(range(len(prices_rolling)), prices_rolling.values, lw=1.5, label=item, alpha=0.7)
ax.set_xlabel("Month"); ax.set_ylabel("Avg Price ($)"); ax.set_title("Price Trends (Top 5 Items)", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 7.6: Price by store
ax = axes[1, 2]
store_prices = {}
for store in stores_sorted:
    store_df = train_sample[train_sample["store_id"] == store]
    store_prices[store] = store_df["sell_price"].mean()
ax.bar(range(len(store_prices)), list(store_prices.values()), color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(store_prices))); ax.set_xticklabels(list(store_prices.keys()))
ax.set_ylabel("Mean Price ($)"); ax.set_title("Avg Price by Store", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("05_prices.png")
plt.close()
print("  ✓ Saved: 05_prices.png")

# ============================================================================
# [8] SPARSITY & ITEM ANALYSIS
# ============================================================================
print("\n[8] SPARSITY & ITEM ANALYSIS...")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 8.1: Item zero-sale distribution
ax = axes[0, 0]
item_zero_pcts = np.array([item_zero[i] / max(item_zero[i] + item_nonzero_days[i], 1) * 100 
                           for i in items_sorted])
ax.hist(item_zero_pcts, bins=50, edgecolor="white", alpha=0.7, color="coral")
ax.axvline(item_zero_pcts.mean(), color="red", ls="--", lw=2, label=f"Mean: {item_zero_pcts.mean():.1f}%")
ax.set_xlabel("Zero Sales %"); ax.set_ylabel("Frequency")
ax.set_title("Distribution of Item Zero-Sale Rates", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 8.2: Items by sparsity category
ax = axes[0, 1]
sparsity_bins = [0, 25, 50, 75, 100]
sparsity_labels = ["<25%", "25-50%", "50-75%", ">75%"]
item_sparsity_cats = pd.cut(item_zero_pcts, bins=sparsity_bins, labels=sparsity_labels, include_lowest=True)
sparsity_counts = item_sparsity_cats.value_counts().sort_index()
colors_sparsity = ["green", "yellow", "orange", "red"]
bars = ax.bar(range(len(sparsity_counts)), sparsity_counts.values, color=colors_sparsity, 
              edgecolor="black", linewidth=1)
ax.set_xticks(range(len(sparsity_counts))); ax.set_xticklabels(sparsity_counts.index)
ax.set_ylabel("Item Count"); ax.set_title("Items by Sparsity Category", fontweight="bold")
for bar, v in zip(bars, sparsity_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + sparsity_counts.max()*0.02, 
            f"{v}\n({v/len(items_sorted)*100:.1f}%)", ha="center", fontsize=9)
ax.grid(alpha=0.3, axis="y")

# 8.3: Item velocity (active days ratio)
ax = axes[0, 2]
item_velocities = np.array([item_nonzero_days[i] / max(item_zero[i] + item_nonzero_days[i], 1) 
                            for i in items_sorted])
ax.hist(item_velocities, bins=50, edgecolor="white", alpha=0.7, color="steelblue")
ax.axvline(item_velocities.mean(), color="red", ls="--", lw=2, label=f"Mean: {item_velocities.mean():.2f}")
ax.set_xlabel("Velocity (fraction of non-zero days)"); ax.set_ylabel("Frequency")
ax.set_title("Item Velocity Distribution", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 8.4: Items per store distribution
ax = axes[1, 0]
items_per_store = np.array([len(store_items[s]) for s in stores_sorted])
ax.bar(range(len(stores_sorted)), items_per_store, color="mediumseagreen", edgecolor="black", linewidth=1)
ax.set_xticks(range(len(stores_sorted))); ax.set_xticklabels(stores_sorted)
ax.set_ylabel("Number of Items"); ax.set_title("Items per Store", fontweight="bold")
ax.grid(alpha=0.3, axis="y")
for i, v in enumerate(items_per_store):
    ax.text(i, v + items_per_store.max()*0.02, str(int(v)), ha="center", fontsize=9)

# 8.5: Store coverage by item
ax = axes[1, 1]
stores_per_item = np.array([len(item_stores[i]) for i in items_sorted])
ax.hist(stores_per_item, bins=n_stores+1, edgecolor="white", alpha=0.7, color="mediumpurple")
ax.axvline(stores_per_item.mean(), color="red", ls="--", lw=2, label=f"Mean: {stores_per_item.mean():.1f}")
ax.set_xlabel("Number of Stores"); ax.set_ylabel("Item Count")
ax.set_title("Store Coverage by Item", fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, axis="y")

# 8.6: Top items by sales
ax = axes[1, 2]
top_items = sorted(item_sales.items(), key=lambda x: x[1], reverse=True)[:15]
item_names, item_sales_vals = zip(*top_items)
ax.barh(range(len(item_names)), item_sales_vals, color="darkorange", edgecolor="black", linewidth=1)
ax.set_yticks(range(len(item_names))); ax.set_yticklabels(item_names)
ax.set_xlabel("Total Sales"); ax.set_title("Top 15 Items by Sales", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
SAVE("06_sparsity.png")
plt.close()
print("  ✓ Saved: 06_sparsity.png")

# ============================================================================
# [9] OUTLIER & ANOMALY DETECTION
# ============================================================================
print("\n[9] OUTLIERS & ANOMALIES...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 9.1: Boxplot of sales (log scale)
ax = axes[0, 0]
sales_nz_log = np.log1p(train_sample[train_sample["sales"] > 0]["sales"])
ax.boxplot([sales_nz_log], labels=["Sales (log)"], patch_artist=True,
           boxprops=dict(facecolor="lightblue", edgecolor="black"),
           whiskerprops=dict(color="black"), capprops=dict(color="black"))
ax.set_ylabel("log(1+sales)"); ax.set_title("Sales Outliers (Boxplot)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 9.2: Daily anomalies (z-score)
ax = axes[0, 1]
daily_sales_clean = daily.sort_values("date")["total_sales"].values
z_scores = np.abs((daily_sales_clean - np.mean(daily_sales_clean)) / np.std(daily_sales_clean))
daily_copy = daily.sort_values("date").copy()
daily_copy["z_score"] = z_scores
daily_copy["anomaly"] = z_scores > 2.5

ax.plot(daily_copy["date"], daily_copy["total_sales"], lw=0.6, color="steelblue", alpha=0.5, label="Daily Sales")
ax.scatter(daily_copy[daily_copy["anomaly"]]["date"], 
          daily_copy[daily_copy["anomaly"]]["total_sales"], 
          color="red", s=50, label="Anomalies (z>2.5)", zorder=5)
ax.set_ylabel("Total Sales"); ax.set_title("Daily Sales Anomalies", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 9.3: Item max sales (top outliers)
ax = axes[1, 0]
item_max_vals = [(i, item_max[i]) for i in items_sorted]
item_max_vals.sort(key=lambda x: x[1], reverse=True)
top_max = item_max_vals[:15]
names, maxs = zip(*top_max)
ax.barh(range(len(names)), maxs, color="crimson", edgecolor="black", linewidth=1)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel("Max Daily Sales"); ax.set_title("Top 15 Items by Max Daily Sales", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# 9.4: Price outliers
ax = axes[1, 1]
price_data = train_sample["sell_price"].dropna()
if len(price_data) > 0:
    Q1, Q3 = price_data.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    price_normal = price_data[(price_data >= lower_bound) & (price_data <= upper_bound)]
    price_outliers = price_data[(price_data < lower_bound) | (price_data > upper_bound)]
    
    ax.scatter(range(len(price_normal)), price_normal.values, alpha=0.3, s=5, color="steelblue", label="Normal")
    ax.scatter([i + len(price_normal) for i in range(len(price_outliers))], price_outliers.values, 
              alpha=0.6, s=20, color="red", label="Outliers")
    ax.axhline(lower_bound, color="orange", ls="--", lw=1, label="Lower bound")
    ax.axhline(upper_bound, color="orange", ls="--", lw=1, label="Upper bound")
    ax.set_xlabel("Record"); ax.set_ylabel("Price ($)"); ax.set_title("Price Outliers (IQR method)", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
SAVE("07_outliers.png")
plt.close()
print("  ✓ Saved: 07_outliers.png")

# ============================================================================
# [10] HIERARCHY & ROLLUP ANALYSIS
# ============================================================================
print("\n[10] HIERARCHY ANALYSIS...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 10.1: State sales composition
ax = axes[0, 0]
state_sales_sorted = sorted(state_sales.items(), key=lambda x: x[1], reverse=True)
states, state_vals = zip(*state_sales_sorted)
colors_state = ["#FF9999", "#66B2FF", "#99FF99"]
ax.pie(state_vals, labels=states, autopct="%1.1f%%", colors=colors_state, startangle=90)
ax.set_title("Sales by State", fontweight="bold", fontsize=12)

# 10.2: Store sales within states
ax = axes[0, 1]
x = np.arange(len(stores_sorted))
width = 0.6
store_colors = {"CA": "red", "TX": "blue", "WI": "green"}
colors = [store_colors.get(s[:2], "gray") for s in stores_sorted]
stores_vals = [store_sales[s] for s in stores_sorted]
ax.bar(x, stores_vals, width, color=colors, edgecolor="black", linewidth=1, alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(stores_sorted)
ax.set_ylabel("Total Sales"); ax.set_title("Sales by Store (colored by state)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

# 10.3: Category sales composition
ax = axes[1, 0]
cat_sales_sorted = sorted(cat_sales.items(), key=lambda x: x[1], reverse=True)
cats, cat_vals = zip(*cat_sales_sorted)
ax.pie(cat_vals, labels=cats, autopct="%1.1f%%", startangle=45)
ax.set_title("Sales by Category", fontweight="bold", fontsize=12)

# 10.4: Department in categories
ax = axes[1, 1]
dept_cat_map = {}
for dept in depts_sorted:
    if dept.startswith("D"):
        cat = dept[0]
        if cat not in dept_cat_map:
            dept_cat_map[cat] = []
        dept_cat_map[cat].append(dept_sales[dept])

cat_labels = sorted(dept_cat_map.keys())
x_pos = np.arange(len(cat_labels))
dept_sales_by_cat = [sum(dept_cat_map[c]) for c in cat_labels]
ax.bar(x_pos, dept_sales_by_cat, color="mediumpurple", edgecolor="black", linewidth=1)
ax.set_xticks(x_pos); ax.set_xticklabels(cat_labels)
ax.set_ylabel("Total Sales"); ax.set_title("Sales by Category (Dept Rollup)", fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("08_hierarchy.png")
plt.close()
print("  ✓ Saved: 08_hierarchy.png")

# ============================================================================
# [11] DATA QUALITY & COMPLETENESS
# ============================================================================
print("\n[11] DATA QUALITY...")

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 11.1: Date coverage
ax = axes[0, 0]
date_range = (train_sample["date"].max() - train_sample["date"].min()).days
expected_rows = date_range * n_items * n_stores
actual_rows = len(train_sample)
coverage = actual_rows / expected_rows * 100 if expected_rows > 0 else 0
ax.text(0.5, 0.7, f"Date Coverage: {coverage:.1f}%", ha="center", fontsize=14, fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8))
ax.text(0.5, 0.5, f"Date Range: {date_range} days", ha="center", fontsize=12)
ax.text(0.5, 0.35, f"Expected rows: {expected_rows:,}\nActual rows: {actual_rows:,}", 
        ha="center", fontsize=11)
ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# 11.2: Missing values per column
ax = axes[0, 1]
missing_cols = dict(sorted(missing_counts.items(), key=lambda x: x[1], reverse=True))
missing_pcts = {k: v / nrows_total * 100 for k, v in missing_cols.items()}
ax.bar(missing_pcts.keys(), missing_pcts.values(), color="coral", edgecolor="black", linewidth=1)
ax.set_ylabel("Missing %"); ax.set_title("Missing Data by Column", fontweight="bold")
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.3, axis="y")

# 11.3: Data types memory usage (rough estimate)
ax = axes[1, 0]
dtypes_memory = {
    "sales (float32)": nrows_total * 4 / 1e6,
    "sell_price (float32)": price_cnt * 4 / 1e6,
    "store_id (category)": n_stores * 10 / 1e6,
    "item_id (category)": n_items * 10 / 1e6,
}
ax.barh(dtypes_memory.keys(), dtypes_memory.values(), color="steelblue", edgecolor="black", linewidth=1)
ax.set_xlabel("Memory (MB)"); ax.set_title("Estimated Memory Usage", fontweight="bold")
ax.grid(alpha=0.3, axis="x")

# 11.4: Data completeness summary
ax = axes[1, 1]
completeness = {
    "Sales": 100 - zero_pct,
    "Prices": price_cnt / nrows_total * 100,
    "Dates": 100,
    "Items": 100,
    "Stores": 100
}
colors_complete = ["red" if v < 80 else "yellow" if v < 95 else "green" for v in completeness.values()]
ax.bar(completeness.keys(), completeness.values(), color=colors_complete, edgecolor="black", linewidth=1)
ax.set_ylabel("Completeness %"); ax.set_title("Data Completeness", fontweight="bold")
ax.set_ylim([0, 105])
ax.axhline(95, color="orange", ls="--", lw=1, label="95% threshold")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
SAVE("09_data_quality.png")
plt.close()
print("  ✓ Saved: 09_data_quality.png")

# ============================================================================
# [12] COMPREHENSIVE SUMMARY REPORT
# ============================================================================
print("\n[12] GENERATING SUMMARY REPORT...")

summary_data = {
    "Category": [
        "VOLUME","VOLUME","VOLUME","VOLUME","VOLUME","VOLUME","VOLUME",
        "DISTRIBUTION","DISTRIBUTION","DISTRIBUTION","DISTRIBUTION","DISTRIBUTION","DISTRIBUTION",
        "SEASONALITY","SEASONALITY","SEASONALITY","SEASONALITY",
        "EVENTS","EVENTS","EVENTS",
        "PRICING","PRICING","PRICING","PRICING",
        "SPARSITY","SPARSITY","SPARSITY","SPARSITY",
        "QUALITY","QUALITY","QUALITY",
        "HIERARCHY","HIERARCHY","HIERARCHY","HIERARCHY"
    ],
    "Metric": [
        "Total Rows","Total Items","Total Stores","Total Categories","Total Departments","Date Range (days)","Total Sales (units)",
        "Mean Sales","Std Dev Sales","Min Sales","Max Sales","Skewness","Coeff of Variation",
        "Best Day of Week","Worst Day of Week","Best Month","Worst Month",
        "Event Lift (%)","Events (Type 1) Count","SNAP Impact (avg)",
        "Mean Price","Price Std Dev","Price-Sales Corr","Price Volatility (top item)",
        "Zero Sales %","Avg Item Zero %","Items with >75% zeros","Avg Item Velocity",
        "Date Coverage %","Price Coverage %","Anomalies (z>2.5)",
        "Stores per Item (avg)","Items per Store (avg)","State with Most Sales","Top Category by Sales"
    ],
    "Value": [
        f"{nrows_total:,}",
        f"{n_items:,}",
        f"{n_stores}",
        f"{n_cats}",
        f"{n_depts}",
        f"{(train_sample['date'].max() - train_sample['date'].min()).days}",
        f"{sales_sum:,.0f}",
        f"{sales_mean:.4f}",
        f"{sales_std:.4f}",
        f"{sales_min:.4f}",
        f"{sales_max:.4f}",
        f"{sales_skew_est:.4f}",
        f"{sales_std/sales_mean:.4f}" if sales_mean > 0 else "N/A",
        dow_labels[np.argmax(dow_vals)],
        dow_labels[np.argmin(dow_vals)],
        labels_m[np.argmax(month_vals)],
        labels_m[np.argmin(month_vals)],
        f"{event_lift:+.2f}%",
        f"{sum(event_type_1_cnt.values()):,}",
        f"{(event_avg / base_avg - 1) * 100:+.2f}%" if base_avg > 0 else "N/A",
        f"{price_mean:.2f}",
        f"{price_std:.2f}",
        f"{corr_coef:.4f}",
        f"{max(item_price_vol.values()) if item_price_vol else 0:.2f}",
        f"{zero_pct:.2f}%",
        f"{items_zero_pct.mean():.2f}%",
        f"{(item_zero_pcts > 75).sum():,}",
        f"{items_velocity.mean():.4f}",
        f"{coverage:.2f}%" if expected_rows > 0 else "N/A",
        f"{price_cnt / nrows_total * 100:.2f}%",
        f"{daily_copy['anomaly'].sum()}",
        f"{stores_per_item.mean():.1f}",
        f"{items_per_store.mean():.0f}",
        sorted(state_sales.items(), key=lambda x: x[1], reverse=True)[0][0],
        sorted(cat_sales.items(), key=lambda x: x[1], reverse=True)[0][0],
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "eda_summary.csv"), index=False)
print(f"  ✓ Saved: eda_summary.csv")

print("\n" + "=" * 90)
print("SUMMARY STATISTICS")
print("=" * 90)
for category in summary_df["Category"].unique():
    print(f"\n  {category}:")
    cat_df = summary_df[summary_df["Category"] == category]
    for _, row in cat_df.iterrows():
        print(f"    {row['Metric']:.<40} {row['Value']:>20}")

# ============================================================================
# [13] ADVANCED QUALITY CHECKS
# ============================================================================
print("\n[13] ADVANCED QUALITY CHECKS...")

dup_count = train_sample.duplicated().sum()
dup_key_count = train_sample.duplicated(subset=["id", "date", "store_id"]).sum()
print(f"  ✓ Duplicates (full row, sample): {dup_count}")
print(f"  ✓ Duplicates (id-date-store, sample): {dup_key_count}")

# ============================================================================
# [14] FEATURE RELATIONSHIPS
# ============================================================================
print("\n[14] FEATURE RELATIONSHIPS...")

num_cols = ["sales", "sell_price", "wday", "month", "year", "snap_CA", "snap_TX", "snap_WI"]
corr_df = train_sample[num_cols].dropna()
corr = corr_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap (numeric features)", fontweight="bold")
SAVE("10_corr_heatmap.png")
plt.close()
print("  ✓ Saved: 10_corr_heatmap.png")

pivot_sc = train_sample.pivot_table(index="store_id", columns="cat_id", values="sales", aggfunc="mean")
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_sc, cmap="YlGnBu")
plt.title("Avg Sales Heatmap: Store x Category", fontweight="bold")
SAVE("11_store_cat_heatmap.png")
plt.close()
print("  ✓ Saved: 11_store_cat_heatmap.png")

# ============================================================================
# [15] STATIONARITY & DECOMPOSITION
# ============================================================================
print("\n[15] STATIONARITY & DECOMPOSITION...")

series = daily_sorted["total_sales"].dropna()

adf_res = adfuller(series, autolag="AIC")
kpss_res = kpss(series, regression="c", nlags="auto")

print(f"  ✓ ADF p-value:  {adf_res[1]:.6f}")
print(f"  ✓ KPSS p-value: {kpss_res[1]:.6f}")

try:
    decomp = seasonal_decompose(series, model="additive", period=7)
    fig = decomp.plot()
    fig.set_size_inches(14, 8)
    plt.tight_layout()
    SAVE("12_seasonal_decompose.png")
    plt.close()
    print("  ✓ Saved: 12_seasonal_decompose.png")
except Exception as e:
    print(f"  ! Decompose failed: {e}")

# ============================================================================
# [16] LAG ANALYSIS
# ============================================================================
print("\n[16] LAG ANALYSIS...")

lags = range(1, 29)
lag_corrs = [series.autocorr(lag=i) for i in lags]

plt.figure(figsize=(10, 4))
plt.bar(lags, lag_corrs, color="steelblue", edgecolor="black")
plt.axhline(0, color="black", lw=0.8)
plt.title("Lag Correlations (1-28)", fontweight="bold")
plt.xlabel("Lag (days)")
plt.ylabel("Correlation")
SAVE("13_lag_corr.png")
plt.close()
print("  ✓ Saved: 13_lag_corr.png")

# ============================================================================
# [17] SPARSE ITEM TIME SERIES
# ============================================================================
print("\n[17] SPARSE ITEM TIME SERIES...")

top_sparse_items = sorted(item_zero.items(), key=lambda x: x[1], reverse=True)[:5]
plt.figure(figsize=(14, 6))
for item_id, _ in top_sparse_items:
    item_ts = train_sample[train_sample["id"] == item_id].groupby("date")["sales"].sum()
    plt.plot(item_ts.index, item_ts.values, lw=1, label=item_id, alpha=0.8)
plt.title("Top Sparse Items - Sales Over Time", fontweight="bold")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
SAVE("14_sparse_items_ts.png")
plt.close()
print("  ✓ Saved: 14_sparse_items_ts.png")

print("\n" + "=" * 90)
print("✓ EDA COMPLETE! (FULL)")
print("=" * 90)
print(f"  Figures saved to: {FIGURES_DIR}")
print(f"  Summary saved to: {OUTPUT_DIR}")
print("=" * 90 + "\n")

In [ ]:
"""
M5 Multivariate Time Series Forecasting - Global Multi-Model Training
=====================================================================
Optimized for Kaggle P100 GPU (16GB VRAM, 13GB RAM)
Models: LightGBM | CatBoost | XGBoost | LSTM
"""

import gc
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import psutil
from pathlib import Path

import lightgbm as lgb
import catboost as cb
import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import mean_squared_error

pd.set_option("display.max_columns", 100)

# ═══════════════════════════════════════════════════════════════════════
# PATHS & CONFIG
# ═══════════════════════════════════════════════════════════════════════
DATA_DIR    = Path("/kaggle/input/datasets/trmnguyntrngbch/dataeda")
OUTPUT_DIR  = Path("/kaggle/working/output")
FIGURES_DIR = Path("/kaggle/working/figures")
MODELS_DIR  = Path("/kaggle/working/models")

for d in [OUTPUT_DIR, FIGURES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HORIZON = 28
LAGS          = [1, 7, 14, 28]
ROLLING_MEANS = [7, 14, 28]
ROLLING_STD   = [7, 28]
ROLLING_SKEW  = []          # bỏ hẳn — tốn RAM, ít giá trị

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def ram():
    m = psutil.virtual_memory()
    g = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"  [MEM] RAM {m.used/1e9:.1f}/{m.total/1e9:.1f} GB  |  GPU {g:.1f} GB")

print("=" * 70)
print("M5 FORECASTING — GLOBAL MULTI-MODEL  (Kaggle P100)")
print(f"Device: {DEVICE}")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════════════════
print("\n[1] LOADING DATA...")

NEEDED_COLS = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "day_id", "date", "wm_yr_wk", "sales", "sell_price",
    "wday", "month", "year",
    "snap_CA", "snap_TX", "snap_WI",
    "event_type_1", "event_type_2",
]

dtypes = {
    "item_id": "category", "dept_id": "category", "cat_id": "category",
    "store_id": "category", "state_id": "category",
    "day_id": "uint16", "wm_yr_wk": "uint16",
    "sales": "float32", "sell_price": "float32",
    "wday": "uint8", "month": "uint8", "year": "uint16",
    "snap_CA": "uint8", "snap_TX": "uint8", "snap_WI": "uint8",
}

train = pd.read_csv(DATA_DIR / "train.csv", dtype=dtypes,
                    usecols=[c for c in NEEDED_COLS if c != "wm_yr_wk"],
                    parse_dates=["date"])
val   = pd.read_csv(DATA_DIR / "val.csv",   dtype=dtypes,
                    usecols=[c for c in NEEDED_COLS if c != "wm_yr_wk"],
                    parse_dates=["date"])

TRAIN_LAST_D = int(train["day_id"].max())
ALL_DAYS     = int(pd.concat([train["day_id"], val["day_id"]]).max())

full = pd.concat([train, val], ignore_index=True)
del train, val
gc.collect()

full = full.sort_values(["id", "day_id"]).reset_index(drop=True)

# Encode id as int để tiết kiệm RAM
full["id_enc"] = full["id"].astype("category").cat.codes.astype("int32")

print(f"  Rows: {full.shape[0]:,} | Cols: {full.shape[1]}")
print(f"  Unique ids: {full['id'].nunique():,}")
print(f"  TRAIN_LAST_D={TRAIN_LAST_D} | ALL_DAYS={ALL_DAYS}")
ram()

# ═══════════════════════════════════════════════════════════════════════
# 2. CALENDAR EVENT ENCODING
# ═══════════════════════════════════════════════════════════════════════
print("\n[2] ENCODING CALENDAR EVENTS...")

EVENT_MAP = {"None": 0, "Cultural": 1, "National": 2, "Religious": 3, "Sporting": 4}
full["event_type_1_enc"] = full["event_type_1"].map(EVENT_MAP).fillna(0).astype("uint8")
full["event_type_2_enc"] = full["event_type_2"].map(EVENT_MAP).fillna(0).astype("uint8")
full["has_event"]        = ((full["event_type_1_enc"] > 0) | (full["event_type_2_enc"] > 0)).astype("uint8")
full["event_type_sum"]   = (full["event_type_1_enc"] + full["event_type_2_enc"]).astype("uint8")
full["snap_CA_event"]    = (full["snap_CA"] * full["has_event"]).astype("uint8")
full["snap_TX_event"]    = (full["snap_TX"] * full["has_event"]).astype("uint8")
full["snap_WI_event"]    = (full["snap_WI"] * full["has_event"]).astype("uint8")

full.drop(columns=["event_type_1", "event_type_2"], inplace=True)
gc.collect()

# ═══════════════════════════════════════════════════════════════════════
# 3. PRICE FEATURES
# ═══════════════════════════════════════════════════════════════════════
print("\n[3] BUILDING PRICE FEATURES...")

price_stats = (
    full.groupby(["item_id", "store_id"])["sell_price"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "price_mean_is", "std": "price_std_is"})
)
full = full.merge(price_stats, on=["item_id", "store_id"], how="left")
del price_stats; gc.collect()

full["price_ratio"]   = (full["sell_price"] / (full["price_mean_is"] + 1e-9)).astype("float32")
full["price_diff"]    = full.groupby(["item_id", "store_id"])["sell_price"].diff().fillna(0).astype("float32")
full["price_changed"] = (full["price_diff"] != 0).astype("uint8")
ram()

# ═══════════════════════════════════════════════════════════════════════
# 4. LAG & ROLLING FEATURES — vectorized (không dùng apply)
# ═══════════════════════════════════════════════════════════════════════
print("\n[4] BUILDING LAG & ROLLING FEATURES (vectorized)...")

g = full.groupby("id")["sales"]

# --- Lags ---
for lag in LAGS:
    full[f"lag_{lag}"] = g.shift(lag).astype("float32")
    print(f"  lag_{lag} ✓")
    gc.collect()

# --- Rolling (shift 1 trước) ---
sales_s1 = g.shift(1)

for w in ROLLING_MEANS:
    full[f"rolling_mean_{w}"] = sales_s1.rolling(w, min_periods=1).mean().astype("float32")
    print(f"  rolling_mean_{w} ✓")
    gc.collect()

for w in ROLLING_STD:
    full[f"rolling_std_{w}"] = sales_s1.rolling(w, min_periods=1).std().fillna(0).astype("float32")
    print(f"  rolling_std_{w} ✓")
    gc.collect()

# --- EWM ---
full["ewm_01"] = sales_s1.ewm(alpha=0.1, min_periods=1).mean().astype("float32")
full["ewm_05"] = sales_s1.ewm(alpha=0.5, min_periods=1).mean().astype("float32")
print("  ewm ✓")

# --- Misc ---
full["lag_1_diff"] = g.shift(1).diff().fillna(0).astype("float32")

is_zero = (full["sales"] == 0).astype("uint8")
full["zero_streak"] = (
    is_zero.groupby(
        (is_zero != is_zero.groupby(full["id"]).shift()).cumsum()
    ).cumsum().astype("uint16")
)

del sales_s1, is_zero, g
gc.collect()
print(f"  Shape after lag/rolling: {full.shape}")
ram()

# ═══════════════════════════════════════════════════════════════════════
# 5. GLOBAL AGGREGATE FEATURES — transform() thay vì merge()
# ═══════════════════════════════════════════════════════════════════════
print("\n[5] BUILDING AGGREGATE FEATURES...")

# transform() ghi thẳng vào full, không tạo DataFrame trung gian lớn
full["store_daily_mean"]  = full.groupby(["store_id","day_id"])["sales"].transform("mean").astype("float32")
gc.collect(); print("  store_daily_mean ✓")

full["store_daily_sum"]   = full.groupby(["store_id","day_id"])["sales"].transform("sum").astype("float32")
gc.collect(); print("  store_daily_sum ✓")

full["dept_daily_mean"]   = full.groupby(["dept_id","day_id"])["sales"].transform("mean").astype("float32")
gc.collect(); print("  dept_daily_mean ✓")

full["cat_daily_mean"]    = full.groupby(["cat_id","day_id"])["sales"].transform("mean").astype("float32")
gc.collect(); print("  cat_daily_mean ✓")

full["state_daily_mean"]  = full.groupby(["state_id","day_id"])["sales"].transform("mean").astype("float32")
gc.collect(); print("  state_daily_mean ✓")

full["global_daily_mean"] = full.groupby("day_id")["sales"].transform("mean").astype("float32")
gc.collect(); print("  global_daily_mean ✓")

full["item_global_mean"]  = full.groupby("item_id")["sales"].transform("mean").astype("float32")
gc.collect(); print("  item_global_mean ✓")

print(f"  Shape: {full.shape}")
ram()

# ═══════════════════════════════════════════════════════════════════════
# 6. PREPARE TRAIN / VAL SPLIT — memmap để không giữ full + array cùng lúc
# ═══════════════════════════════════════════════════════════════════════
print("\n[6] PREPARING TRAIN / VAL DATA...")

DROP_COLS    = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "day_id", "date"]
FEATURE_COLS = [c for c in full.columns if c not in DROP_COLS + ["sales"]]
N_FEAT = len(FEATURE_COLS)
print(f"  Features: {N_FEAT}")

min_day  = max(LAGS) + 1
tr_idx   = full.index[(full["day_id"] >= min_day) & (full["day_id"] <= TRAIN_LAST_D)]
val_idx  = full.index[(full["day_id"] > TRAIN_LAST_D) & (full["day_id"] <= ALL_DAYS)]
N_TR, N_VAL = len(tr_idx), len(val_idx)
print(f"  Train rows: {N_TR:,} | Val rows: {N_VAL:,}")

# Tạo memmap files trên disk — không chiếm RAM
MMAP_DIR = Path("/kaggle/working/mmap")
MMAP_DIR.mkdir(exist_ok=True)

mm_X_tr  = np.memmap(MMAP_DIR/"X_tr.dat",  dtype="float32", mode="w+", shape=(N_TR,  N_FEAT))
mm_y_tr  = np.memmap(MMAP_DIR/"y_tr.dat",  dtype="float32", mode="w+", shape=(N_TR,))
mm_X_val = np.memmap(MMAP_DIR/"X_val.dat", dtype="float32", mode="w+", shape=(N_VAL, N_FEAT))
mm_y_val = np.memmap(MMAP_DIR/"y_val.dat", dtype="float32", mode="w+", shape=(N_VAL,))

# Ghi từng cột vào memmap — full vẫn còn nhưng chỉ đọc 1 col mỗi lần
print("  Writing train memmap col-by-col...")
for i, col in enumerate(FEATURE_COLS):
    mm_X_tr[:, i] = full[col].iloc[tr_idx].to_numpy(dtype=np.float32, na_value=0.0)
    if i % 10 == 0:
        mm_X_tr.flush(); gc.collect()
        print(f"    col {i}/{N_FEAT}", end="\r")
mm_y_tr[:] = full["sales"].iloc[tr_idx].to_numpy(dtype=np.float32)
mm_X_tr.flush(); mm_y_tr.flush()
print(f"\n  Train memmap done ✓")
ram()

print("  Writing val memmap col-by-col...")
for i, col in enumerate(FEATURE_COLS):
    mm_X_val[:, i] = full[col].iloc[val_idx].to_numpy(dtype=np.float32, na_value=0.0)
    if i % 10 == 0:
        mm_X_val.flush(); gc.collect()
        print(f"    col {i}/{N_FEAT}", end="\r")
mm_y_val[:] = full["sales"].iloc[val_idx].to_numpy(dtype=np.float32)
mm_X_val.flush(); mm_y_val.flush()
print(f"\n  Val memmap done ✓")

# Xóa full NGAY sau khi ghi xong
del full; gc.collect()
ram()

# Load lại memmap read-only — chỉ tốn virtual address, không copy vào RAM
X_tr  = np.memmap(MMAP_DIR/"X_tr.dat",  dtype="float32", mode="r", shape=(N_TR,  N_FEAT))
y_tr  = np.memmap(MMAP_DIR/"y_tr.dat",  dtype="float32", mode="r", shape=(N_TR,))
X_val = np.memmap(MMAP_DIR/"X_val.dat", dtype="float32", mode="r", shape=(N_VAL, N_FEAT))
y_val = np.memmap(MMAP_DIR/"y_val.dat", dtype="float32", mode="r", shape=(N_VAL,))

print(f"  X_tr: {X_tr.shape} | X_val: {X_val.shape}")
ram()

# ═══════════════════════════════════════════════════════════════════════
# 7. LIGHTGBM
# ═══════════════════════════════════════════════════════════════════════
print("\n[7] TRAINING LIGHTGBM...")

lgb_params = dict(
    objective="regression", metric="rmse",
    boosting_type="gbdt", learning_rate=0.05,
    num_leaves=128, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1,
    seed=42, device="gpu", gpu_platform_id=0, gpu_device_id=0,
    verbose=-1,
)

dtrain = lgb.Dataset(X_tr,  label=y_tr,  free_raw_data=False)
dval   = lgb.Dataset(X_val, label=y_val, free_raw_data=False, reference=dtrain)

lgb_model = lgb.train(
    lgb_params, dtrain,
    num_boost_round=1000,
    valid_sets=[dval],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
lgb_pred = lgb_model.predict(X_val)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_pred))
print(f"  LightGBM RMSE: {lgb_rmse:.4f}")
lgb_model.save_model(str(MODELS_DIR / "lgb.txt"))
del dtrain, dval; gc.collect()
ram()

# ═══════════════════════════════════════════════════════════════════════
# 8. CATBOOST
# ═══════════════════════════════════════════════════════════════════════
print("\n[8] TRAINING CATBOOST...")

cb_model = cb.CatBoostRegressor(
    loss_function="RMSE", eval_metric="RMSE",
    iterations=1000, learning_rate=0.05, depth=8,
    task_type="GPU", devices="0",
    random_seed=42, verbose=100,
    early_stopping_rounds=50,
)
cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
cb_pred = cb_model.predict(X_val)
cb_rmse = np.sqrt(mean_squared_error(y_val, cb_pred))
print(f"  CatBoost RMSE: {cb_rmse:.4f}")
cb_model.save_model(str(MODELS_DIR / "catboost.cbm"))
del cb_model; gc.collect()
ram()

# ═══════════════════════════════════════════════════════════════════════
# 9. XGBOOST
# ═══════════════════════════════════════════════════════════════════════
print("\n[9] TRAINING XGBOOST...")

xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05,
    max_depth=8, subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="gpu_hist", device="cuda",
    random_state=42, early_stopping_rounds=50,
    eval_metric="rmse", verbosity=1,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
xgb_pred = xgb_model.predict(X_val)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
print(f"  XGBoost RMSE: {xgb_rmse:.4f}")
xgb_model.save_model(str(MODELS_DIR / "xgb.json"))
del xgb_model; gc.collect()
ram()

# ═══════════════════════════════════════════════════════════════════════
# 10. LSTM
# ═══════════════════════════════════════════════════════════════════════
print("\n[10] TRAINING LSTM...")

SEQ_LEN    = 28
HIDDEN     = 128
N_LAYERS   = 2
DROPOUT    = 0.2
LR         = 1e-3
BATCH_SIZE = 1024
EPOCHS     = 15
N_FEATURES = X_tr.shape[1]

class SalesLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(N_FEATURES, HIDDEN, N_LAYERS,
                            batch_first=True, dropout=DROPOUT)
        self.head = nn.Sequential(
            nn.Linear(HIDDEN, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

def make_sequences(X, y, seq_len):
    """
    Dùng DataLoader lazy thay vì tạo toàn bộ sequence array trước.
    Với memmap, sliding_window_view sẽ copy hết vào RAM → dùng Dataset custom.
    """
    class SeqDataset(torch.utils.data.Dataset):
        def __init__(self, X, y, seq_len):
            self.X = X
            self.y = y
            self.seq_len = seq_len
            self.n = len(y) - seq_len

        def __len__(self):
            return self.n

        def __getitem__(self, i):
            x = torch.tensor(self.X[i:i+self.seq_len], dtype=torch.float32)
            y = torch.tensor(self.y[i+self.seq_len],   dtype=torch.float32)
            return x, y

    return SeqDataset(X, y, seq_len)

print("  Building sequence datasets (lazy)...")
tr_seq_ds  = make_sequences(X_tr,  y_tr,  SEQ_LEN)
val_seq_ds = make_sequences(X_val, y_val, SEQ_LEN)

tr_loader  = DataLoader(tr_seq_ds,  batch_size=BATCH_SIZE, shuffle=True,  pin_memory=False, num_workers=0)
val_loader = DataLoader(val_seq_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False, num_workers=0)

model     = SalesLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)
criterion = nn.MSELoss()

best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    tr_loss = 0
    for xb, yb in tr_loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item()
    tr_loss /= len(tr_loader)

    # --- val ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            val_loss += criterion(model(xb), yb).item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)
    print(f"  Epoch {epoch:02d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODELS_DIR / "lstm_best.pt")

# Eval LSTM
model.load_state_dict(torch.load(MODELS_DIR / "lstm_best.pt"))
model.eval()
preds_lstm = []
with torch.no_grad():
    for xb, _ in val_loader:
        preds_lstm.append(model(xb.to(DEVICE)).cpu().numpy())
lstm_pred = np.concatenate(preds_lstm)
print(f"  LSTM inference done, samples: {len(lstm_pred):,}")
ram()

# ═══════════════════════════════════════════════════════════════════════
# 11. ENSEMBLE
# ═══════════════════════════════════════════════════════════════════════
print("\n[11] ENSEMBLE...")

# LSTM ngắn hơn SEQ_LEN rows so với các model khác → align
n = len(lstm_pred)
ens_pred = (
    0.35 * lgb_pred[-n:] +
    0.25 * cb_pred[-n:]  +
    0.25 * xgb_pred[-n:] +
    0.15 * lstm_pred
)

# ═══════════════════════════════════════════════════════════════════════
# 12. METRICS — MAE, RMSE, WAPE, Bias(%), MASE, R²
# ═══════════════════════════════════════════════════════════════════════
print("\n[12] COMPUTING METRICS...")

from sklearn.metrics import r2_score

def compute_metrics(y_true, y_pred, y_train_naive=None, name="Model"):
    """
    y_train_naive: dùng để tính MASE — cần chuỗi train gốc (naive forecast = lag-1)
                   Nếu None → MASE = N/A
    """
    y_true = np.array(y_true, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)

    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    wape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-9) * 100
    bias = np.mean(y_pred - y_true) / (np.mean(y_true) + 1e-9) * 100
    r2   = r2_score(y_true, y_pred)

    if y_train_naive is not None:
        naive_mae = np.mean(np.abs(np.diff(y_train_naive.astype(np.float64))))
        mase = mae / (naive_mae + 1e-9)
    else:
        mase = float("nan")

    return {
        "Model":    name,
        "MAE":      round(mae,  4),
        "RMSE":     round(rmse, 4),
        "WAPE(%)":  round(wape, 4),
        "Bias(%)":  round(bias, 4),
        "MASE":     round(mase, 6) if not np.isnan(mase) else "N/A",
        "R2":       round(r2,   4),
    }

# y_train_naive cho MASE — dùng y_tr (sales chuỗi train theo thứ tự)
# Lưu ý: y_tr đã được sort theo [id, day_id] nên naive diff hợp lệ
y_train_naive = y_tr  # shape (N,) — naive = shift 1

rows = []

# LightGBM, CatBoost, XGBoost → y_val đầy đủ
rows.append(compute_metrics(y_val,     lgb_pred,  y_train_naive, "LightGBM"))
rows.append(compute_metrics(y_val,     cb_pred,   y_train_naive, "CatBoost"))
rows.append(compute_metrics(y_val,     xgb_pred,  y_train_naive, "XGBoost"))

# LSTM → y_val[-n:] vì bị ngắn SEQ_LEN rows đầu
rows.append(compute_metrics(y_val[-n:], lstm_pred, None,          "LSTM"))

# Ensemble
rows.append(compute_metrics(y_val[-n:], ens_pred,  None,          "Ensemble"))

metrics_df = pd.DataFrame(rows).set_index("Model")

# ═══════════════════════════════════════════════════════════════════════
# 13. SUMMARY & PLOT
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# Save CSV
metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv")
print(f"\n  Metrics saved → {OUTPUT_DIR / 'model_metrics.csv'}")

# ── Plot 1: Bar chart so sánh RMSE & MAE ──────────────────────────────
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
models = metrics_df.index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("M5 Forecasting — Model Comparison", fontsize=14, fontweight="bold")

# RMSE
ax = axes[0]
vals = metrics_df["RMSE"].astype(float).values
bars = ax.bar(models, vals, color=COLORS)
ax.set_title("RMSE (lower = better)")
ax.set_ylabel("RMSE")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}",
            ha="center", va="bottom", fontsize=8)
ax.tick_params(axis="x", rotation=20)

# MAE
ax = axes[1]
vals = metrics_df["MAE"].astype(float).values
bars = ax.bar(models, vals, color=COLORS)
ax.set_title("MAE (lower = better)")
ax.set_ylabel("MAE")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}",
            ha="center", va="bottom", fontsize=8)
ax.tick_params(axis="x", rotation=20)

# R²
ax = axes[2]
vals = metrics_df["R2"].astype(float).values
bars = ax.bar(models, vals, color=COLORS)
ax.set_title("R² (higher = better)")
ax.set_ylabel("R²")
ax.set_ylim(0, 1.05)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}",
            ha="center", va="bottom", fontsize=8)
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "metrics_comparison.png", dpi=120)
print(f"  Plot saved → {FIGURES_DIR / 'metrics_comparison.png'}")

# ── Plot 2: WAPE & Bias ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("M5 Forecasting — WAPE & Bias", fontsize=13, fontweight="bold")

ax = axes[0]
vals = metrics_df["WAPE(%)"].astype(float).values
bars = ax.bar(models, vals, color=COLORS)
ax.set_title("WAPE % (lower = better)")
ax.set_ylabel("WAPE (%)")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.05, f"{v:.2f}%",
            ha="center", va="bottom", fontsize=8)
ax.tick_params(axis="x", rotation=20)

ax = axes[1]
vals = metrics_df["Bias(%)"].astype(float).values
colors_bias = ["#e74c3c" if v > 0 else "#2ecc71" for v in vals]
bars = ax.bar(models, vals, color=colors_bias)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Bias % (closer to 0 = better)")
ax.set_ylabel("Bias (%)")
for bar, v in zip(bars, vals):
    offset = 0.02 if v >= 0 else -0.08
    ax.text(bar.get_x() + bar.get_width()/2, v + offset, f"{v:.2f}%",
            ha="center", va="bottom", fontsize=8)
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "wape_bias.png", dpi=120)
print(f"  Plot saved → {FIGURES_DIR / 'wape_bias.png'}")

print("\nDone!")

M5 FORECASTING — GLOBAL MULTI-MODEL  (Kaggle P100)
Device: cuda

[1] LOADING DATA...
  Rows: 58,327,370 | Cols: 19
  Unique ids: 30,490
  TRAIN_LAST_D=1885 | ALL_DAYS=1913
  [MEM] RAM 8.0/33.7 GB  |  GPU 0.0 GB

[2] ENCODING CALENDAR EVENTS...

[3] BUILDING PRICE FEATURES...
  [MEM] RAM 8.4/33.7 GB  |  GPU 0.0 GB

[4] BUILDING LAG & ROLLING FEATURES (vectorized)...
  lag_1 ✓
  lag_7 ✓
  lag_14 ✓
  lag_28 ✓
  rolling_mean_7 ✓
  rolling_mean_14 ✓
  rolling_mean_28 ✓
  rolling_std_7 ✓
  rolling_std_28 ✓
  ewm ✓
  Shape after lag/rolling: (58327370, 42)
  [MEM] RAM 11.4/33.7 GB  |  GPU 0.0 GB

[5] BUILDING AGGREGATE FEATURES...
  store_daily_mean ✓
  store_daily_sum ✓
  dept_daily_mean ✓
  cat_daily_mean ✓
  state_daily_mean ✓
  global_daily_mean ✓
  item_global_mean ✓
  Shape: (58327370, 49)
  [MEM] RAM 13.0/33.7 GB  |  GPU 0.0 GB

[6] PREPARING TRAIN / VAL DATA...
  Features: 40
  Train rows: 56,619,930 | Val rows: 853,720
  Writing train memmap col-by-col...
    col 30/40
  Train memmap

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 50 rounds
